# AI Marks Remover — GPU Experiment (iPhone)

**Experimental, not an official SynthID remover.** Works on images you own. The output may still be detectable as AI-generated or carry a surviving watermark. Models are large and free Colab GPU sessions may run out of memory or storage.

Your photo stays in the Colab runtime during processing; it is **not private on your phone**. Do not upload sensitive photos.

Use **Runtime → Change runtime type → T4 GPU** if available, then run each cell in order. Free GPU availability is not guaranteed.


In [ ]:
import sys, subprocess, shutil, os, torch
print('Python:',sys.version.split()[0])
print('GPU available:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU allocated. In Colab: Runtime > Change runtime type > T4 GPU. Free availability is not guaranteed.')
print('GPU:',torch.cuda.get_device_name(0))
print('GPU memory (GB):',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))
print('Disk free (GB):',round(shutil.disk_usage('/content').free/1024**3,1))
print('Note: The selected model can exceed a free T4 GPU memory/disk budget.')


## 1 — Install upstream package
This downloads open-source dependencies. It can take several minutes and substantial disk space.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','remove-ai-watermarks[qwen-zimage]'],check=True)
subprocess.run(['remove-ai-watermarks','invisible','--help'],check=True)


## 2 — Upload one image
Only upload a photo you own and are comfortable storing temporarily in a Google Colab session. Supported input depends on the upstream tool.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError('Upload exactly one image')
filename=next(iter(uploaded))
if Path(filename).suffix.lower() not in {'.png','.jpg','.jpeg','.webp'}: raise ValueError('Use PNG, JPEG or WebP')
input_path=Path('/content/input'+Path(filename).suffix.lower())
shutil.move(filename,input_path)
output_path=Path('/content/processed.png')
print('Input:',input_path)


## 3 — Regenerate (experimental)
This changes image pixels and may require GPU memory beyond a free T4. A successful file does **not** prove SynthID was removed. Never rely on the tool's own score as Google's official result.


In [ ]:
import subprocess
cmd=['remove-ai-watermarks','invisible',str(input_path),'-o',str(output_path),'--force','--cpu-offload']
print('Running:', ' '.join(cmd))
subprocess.run(cmd,check=True,timeout=3600)
if not output_path.is_file() or output_path.stat().st_size==0:
    raise RuntimeError('The model did not produce an output image')
print('Output created:',round(output_path.stat().st_size/1024/1024,2),'MB')


## 4 — Download
Check image quality and any provider watermark verdict independently; do not interpret an image export as verified removal.


In [ ]:
from google.colab import files
if not output_path.is_file(): raise RuntimeError('No processed image available. Run the previous cell successfully first.')
files.download(str(output_path))
